In [2]:
from scipy.io import loadmat    # used to turn .mat files into numpy Arrays
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [3]:
def plot_ABP_PPG_signal(ABP, PPG, fs):

    t = np.arange(len(PPG)) / fs    # time axis, in seconds 
    
    plt.figure(figsize=(12,6))

    plt.subplot(2,1,1)
    plt.plot(t, ABP, color='blue')
    plt.title('ABP Signal')
    plt.xlabel('Time [s]')
    plt.ylabel('Normalized ABP ([0,1])')

    plt.subplot(2,1,2)
    plt.plot(t, PPG, color='red')
    plt.title('PPG Signal')
    plt.xlabel('Time [s]')
    plt.ylabel('Light Intensity [V]')

    plt.tight_layout()

    plt.show()

In [4]:
def unwrap_matlab_array(x):
    '''
    Recursively unwrap MATLAB object arrays to a plain Python number
    '''
    x = np.asarray(x)
    while x.dtype == object:
        x = x[0]
    if np.size(x) == 1:
        return x.item()
    return x


In [ ]:
data_path = 'ABP_PPG'
data_file = sorted(os.listdir(data_path))
rows = []
print(len(data_file))
for i in range(1000):
    file_path = os.path.join(data_path, data_file[i])
    data = loadmat(file_path)
    for j in range(1):
        # print(f'accessing struct {j}:')
        print(f'Accessing datapoint {i}')
        sp = data['signal_processing'][0,j]
        segment = sp['signal']

        ABP = segment[0]

        beats = sp['struct_features']['beats_abp'][0,0]
        beats_masked = np.where(beats == 0, np.nan, beats)  # Replace 0 padding with NaN
        DBP = np.nanmin(beats_masked, axis=1).mean()
        SBP = np.nanmax(beats_masked, axis=1).mean()
        
        '''
        # mean_pulse_abp aligns all the beats first and then calculates the mean values for each point
        beats_mean = sp['struct_features']['mean_pulse_abp'][0]
        # print(beats_mean)
        beats_mean = np.asarray(beats_mean[0], dtype=float)
        beats_mean_masked = np.where(beats_mean == 0, np.nan, beats_mean)
        # print(beats_mean_masked)
        DBP_mean = np.nanmin(beats_mean_masked, axis=1)
        SBP_mean = np.nanmax(beats_mean_masked, axis=1)
        print(f'means:{DBP_mean, SBP_mean}')

        #mean(min(beat)) != min(mean(beat))
        #mean(max(beat)) != max(mean(beat))
        '''
        
        feat_ppg = sp['struct_stats_features']['ppg'][0,0]

        ppg_mean_dict = {}
        ppg_std_dict = {}
        for stat_name, target_dict in [('mean', ppg_mean_dict),('std', ppg_std_dict)]:
            stat_struct = feat_ppg[stat_name][0,0]
            for field in stat_struct.dtype.names:
                target_dict[f'ppg_{stat_name}_{field}'] = unwrap_matlab_array(stat_struct[field])

        row = {
            'data ID': data_file[i],
            'struct': j,
            'DBP': DBP,
            'SBP': SBP
        }
        row.update(ppg_mean_dict)
        rows.append(row)

        df = pd.DataFrame(rows)
        df.to_csv('ABP_PPG_features.csv', index=False)


        '''
        delineator_abp = sp['struct_delineator']['ABP'][0,0]
        print(delineator_abp.dtype.names)
        sysPeak_idx = delineator_abp['SysPeak'][0,0]
        foot_idx = delineator_abp['foot'][0,0]
        

        # numpy.ndaaray's of the indices
        sysPeak_idx = sysPeak_idx[0]
        foot_idx = foot_idx[0]

        SBP = ABP[sysPeak_idx]
        DBP = ABP[foot_idx]
        SBP_mean = SBP.mean()
        DBP_mean = DBP.mean()
        
        feat = sp['struct_stats_features']['ppg'][0,0]
        print(feat.dtype.names)
        feat_mean = feat['mean'][0,0]
        feat_std = feat['std'][0,0]
        print(feat_mean)
        '''
        
        

5857
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struct 0:
accessing struc

KeyboardInterrupt: 